In [1]:
print("ok")

ok


In [2]:
2+2

4

In [3]:
%pwd

'f:\\LLMMedical\\research'

In [4]:
import os
os.chdir("../")

In [6]:
%pwd

'f:\\LLMMedical'

In [7]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [8]:
#Extract Data From the PDF File
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader)

    documents=loader.load()

    return documents

In [9]:
extracted_data=load_pdf_file(data='Data/')

Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 38 0 (offset 0)
Ignoring wrong pointing object 42 0 (offset 0)
Ignoring wrong pointing object 46 0 (offset 0)
Ignoring wrong pointing object 50 0 (offset 0)
Ignoring wrong pointing object 54 0 (offset 0)
Ignoring wrong pointing object 59 0 (offset 0)
Ignoring wrong pointing object 63 0 (offset 0)
Ignoring wrong pointing object 67 0 (offset 0)
Ignoring wrong pointing object 71 0 (offset 0)
Ignoring wrong pointing object 76 0 (offset 0)
Ignoring wrong pointing object 80 0 (offset 0)
Ignoring wrong pointing object 84 0 (offset 0)
Ignoring wrong pointing object 88 0 (offset 0)
Ignoring wrong pointing object 109 0 (offset 0)
Ignoring wrong pointing object 113 0 (offset 0)
Ignoring wrong pointing object 117 0 (offset 0)
Ignoring wrong pointing object 121 0 (offset 0)
Ignoring wrong pointing object 125 0 (offset 0)
Ignoring 

In [10]:
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [11]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks", len(text_chunks))

Length of Text Chunks 30022


In [12]:
from langchain.embeddings import HuggingFaceEmbeddings

In [15]:
#Download the Embeddings from Hugging Face
def download_hugging_face_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [20]:
pip install --upgrade huggingface_hub


Note: you may need to restart the kernel to use updated packages.


In [21]:
pip install --upgrade sentence-transformers


  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 2.2.2
    Uninstalling sentence-transformers-2.2.2:
      Successfully uninstalled sentence-transformers-2.2.2
Note: you may need to restart the kernel to use updated packages.


In [22]:
import sentence_transformers


In [24]:
pip install huggingface_hub[hf_xet]


   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   -- ------------------------------------- 0.3/4.2 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/4.2 MB 3.4 MB/s eta 0:00:02
   --------------- ------------------------ 1.6/4.2 MB 4.0 MB/s eta 0:00:01
   ------------------------- -------------- 2.6/4.2 MB 3.6 MB/s eta 0:00:01
   ------------------------------ --------- 3.1/4.2 MB 3.5 MB/s eta 0:00:01
   ------------------------------------- -- 3.9/4.2 MB 3.4 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 3.4 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


In [29]:
!pip install hf_xet



In [31]:

embeddings = download_hugging_face_embeddings()

In [32]:
query_result = embeddings.embed_query("Hello world")
print("Length", len(query_result))

Length 384


In [45]:
from dotenv import load_dotenv
load_dotenv()

True

In [46]:
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')
OPENAI_API_KEY=os.environ.get('OPENAI_API_KEY')

In [ ]:
from pinecone.grpc import PineconeGRPC as Pinecone
from pinecone import ServerlessSpec
import os

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medicalbot"


pc.create_index(
    name=index_name,
    dimension=384, 
    metric="cosine", 
    spec=ServerlessSpec(
        cloud="aws", 
        region="us-east-1"
    ) 
) 

In [47]:
import os
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

In [39]:
# Embed each chunk and upsert the embeddings into your Pinecone index.
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name=index_name,
    embedding=embeddings, 
)

In [48]:
# Load Existing index 

from langchain_pinecone import PineconeVectorStore
# Embed each chunk and upsert the embeddings into your Pinecone index.
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [ ]:
docsearch

In [49]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [52]:
retrieved_docs = retriever.invoke("What are the causes of primary DUB?")

In [53]:
retrieved_docs

[Document(id='e20d722c-1262-4df4-8c0f-82a764f5a291', metadata={'creationdate': '2020-04-24T02:58:46+06:00', 'creator': 'Adobe Scan for Android 20.03.09', 'moddate': '2020-04-24T12:03:15+06:00', 'page': 95.0, 'page_label': '96', 'producer': 'Adobe Acrobat Pro DC 18 Paper Capture Plug-in', 'source': 'Data\\Endeavour Medicine Paper II.pdf', 'title': '', 'total_pages': 617.0}, page_content='I) Primary syph i I is (chancre) \n2) Genital herpes (most common cause)'),
 Document(id='87ff87f9-e1b7-4f21-a8e2-382ca215e9a6', metadata={'creationdate': '2017-03-17T11:32:08+02:00', 'creator': 'pdfsam-console (Ver. 2.4.1e)', 'moddate': '2017-06-23T15:42:08+07:00', 'page': 708.0, 'page_label': '709', 'producer': 'iText 2.1.7 by 1T3XT', 'source': 'Data\\ROBBINS BASIC PATHOLOGY.pdf', 'total_pages': 910.0}, page_content='inoculation site. This widespread dissemination accounts \nfor the protean manifestations of the disease (Fig. 18.19), \nwhich in adults can be divided into primary, secondary, \nand tert

In [54]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.4, max_tokens=500)

In [55]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate


system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)

In [56]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [65]:
response = rag_chain.invoke({"input": "What are the causes of primary DUB?"})
print(response["answer"])


System: The most common cause of primary DUB is genital herpes, which can lead to widespread dissemination and protean manifestations of the disease. Another cause is inoculation at the site of infection, which can result in primary, secondary, and tertiary stages of the disease. Lastly, systemic dissemination of organisms and immune response can also contribute to primary DUB.


In [66]:
response = rag_chain.invoke({"input": "What is the main reason?"})
print(response["answer"])



The main reason is to relieve respiratory distress.
